# llm

> One call shape for each model: `ask(prompt, sp) -> str`.

This module connects the tutor to a model. The tutor needs only one function, `ask(prompt, sp)`, which returns text.
The module also parses JSON replies and gives `Fake`, a model for tests that runs offline.

In [ ]:
#| default_exp llm

In [ ]:
#| hide
from fastcore.test import *

In [ ]:
#| hide
import os, tempfile
os.environ['VISHALAKSHI_VAULT'] = os.environ['ABHYASA_VAULT'] = os.path.join(tempfile.mkdtemp(), 'vault.db')   # a test that opens the default vault gets a temporary one

Import the tools that this module uses.

In [ ]:
#| export
import json, os, re
from fastcore.utils import *

## The model

Each surface gives its own `ask`.

| surface | `ask` |
|---|---|
| web app, notebook | `rishi_ask`: a new `Chat` of the model layer for each call, on `$ABHYASA_MODEL` (default `claude/sonnet`) |
| the ramabana terminal agent | `agent.oneshot`, so the routing of the agent applies |
| tests | `Fake` |

In [ ]:
#| export
DFLT_MODEL = 'claude/sonnet'

def rishi_ask(model:str=None):
    "An `ask(prompt, sp='')` on a `rishi.Chat` (the model layer) for `model`, else `$ABHYASA_MODEL`, else `DFLT_MODEL`."
    from rishi import Chat, resp_text
    model = model or os.getenv('ABHYASA_MODEL') or DFLT_MODEL
    def ask(prompt, sp=''):
        chat = Chat(model, sp=sp)
        try: return resp_text(chat(prompt))
        finally: chat.close()
    ask.model = model
    return ask

## JSON replies

`parse_json` finds the JSON in a reply. `ask_json` asks again one time when the reply is not valid JSON.

In [ ]:
#| export
def parse_json(s:str):
    "The first JSON object or array in `s`, also inside a ``` fence. Raise `ValueError` when there is none."
    s = s.strip()
    dec = json.JSONDecoder()
    # the whole reply first, then each fence, then the first object or array in the text: a fence inside a JSON string stays whole
    for t in [s, *(m.strip() for m in re.findall(r'```(?:json)?\s*(.*?)```', s, re.S))]:
        try: return json.loads(t)
        except json.JSONDecodeError: pass
    for i, ch in enumerate(s):
        if ch in '{[':
            try: return dec.raw_decode(s, i)[0]
            except json.JSONDecodeError: continue
    raise ValueError(f'no JSON in the reply: {s[:200]!r}')

def ask_json(ask, prompt:str, sp:str='', tries:int=2):
    "Ask for JSON and parse it. On a bad reply, ask one more time with the parse error."
    p = prompt + '\n\nReply with only one JSON value and no other text.'
    for i in range(tries):
        out = ask(p, sp)
        try: return parse_json(out)
        except ValueError as e:
            if i == tries - 1: raise
            p = f'{prompt}\n\nYour last reply was not valid JSON ({e}). Reply with only one JSON value.'

`parse_json` reads JSON in a fence, JSON inside other text, and arrays. A fence inside a JSON string does not break it. It fails when there is no JSON.

In [ ]:
test_eq(parse_json('```json\n{"a": 1}\n```'), {'a': 1})
test_eq(parse_json('Here it is: {"a": [1, 2]} thanks'), {'a': [1, 2]})
test_eq(parse_json('[{"x": 1}]'), [{'x': 1}])
test_fail(lambda: parse_json('nothing'), contains='no JSON')
test_eq(parse_json('{"answer": "Use ```python\\nx = 1\\n``` here."}'), {'answer': 'Use ```python\nx = 1\n``` here.'})
test_eq(parse_json('Sure. {"a": 1} and then {"b": 2}'), {'a': 1})

## A model for tests

`Fake` replies from a dict of rules. The first rule with a key in the prompt gives the reply.
It sends a dict or a list reply as JSON. It keeps each call in `calls`.

In [ ]:
#| export
class Fake:
    "A stand-in model for tests: `Fake({'grade': {...}, 'question': {...}})`."
    def __init__(self, rules:dict=None, dflt='{}'): self.rules, self.dflt, self.calls = rules or {}, dflt, []
    def __call__(self, prompt, sp=''):
        self.calls.append((prompt, sp))
        r = next((v for k, v in self.rules.items() if k in prompt or k in sp), self.dflt)
        if callable(r): r = r(prompt, sp)
        return r if isinstance(r, str) else json.dumps(r)

A matching rule gives its reply, and `Fake` records the call.

In [ ]:
f = Fake({'GRADE': dict(correct=True)})
test_eq(ask_json(f, 'GRADE this'), dict(correct=True))
test_eq(len(f.calls), 1)

After a reply that is not JSON, `ask_json` asks again and gets the JSON.

In [ ]:
bad = Fake(dict(X=lambda p, sp: 'oops' if 'not valid' not in p else '{"ok": 1}'))
test_eq(ask_json(bad, 'X'), {'ok': 1})

## A model for each task

The tutor does five tasks, and each task can use its own model. A quick model can grade and give hints. A strong model can
write questions and lessons. The Settings page saves the choice. `Router` reads it on each call, so a change has an effect on the next call.

| task | what the model does |
|---|---|
| `question` | writes a question and its hidden model answer |
| `grade` | reads the conversation and grades it |
| `hint` | gives the next hint |
| `lesson` | teaches the smallest step after a miss |
| `extract` | finds the concepts in a text |

In [ ]:
#| export
TASKS = ('question', 'grade', 'hint', 'lesson', 'extract')

`model_ask` gives an `ask` for any model name. Install the `models` extra (`pip install abhyasa[models]`) to use the full list of models.
With the extra, the name can be any model in that list: Claude Code, Codex, OpenAI, Copilot, Gemini, or a model on this machine (LiteRT, mlx,
llama.cpp, Ollama). The extra finds the model for the name and gives a backend of the model layer. Without the extra, the name goes to the model layer as it is.
Each model has one lock, so the calls take turns. A backend from the full list refuses a second call while one call runs.

In [ ]:
#| export
import threading, time
_asks, _asks_lock = {}, threading.Lock()

def _ramabana_ask(name):
    from ramabana.core import resolve
    from ramabana.runtime import make_backend
    b, lock = make_backend(resolve(name)), threading.Lock()
    def ask(prompt, sp=''):
        with lock: out = b.oneshot(prompt, sp)
        if not out and getattr(b, 'note', ''): raise RuntimeError(b.note)
        return out
    return ask

def model_ask(name:str):
    "An `ask(prompt, sp='')` for model `name`, made once and kept."
    with _asks_lock:
        if name not in _asks:
            try: _asks[name] = _ramabana_ask(name)
            except ImportError: _asks[name] = rishi_ask(name)
            _asks[name].model = name
        return _asks[name]

## Names for the model list

The Settings page shows a list of models. Each item has a name that people know, such as "Claude Sonnet" or "GPT-5.5".
The model id, such as `claude/sonnet`, is the value that the settings keep. `model_label` makes the name from the id.
`PROVIDERS` gives the name of each group in the list.

In [ ]:
#| export
PROVIDERS = dict(claude='Claude', anthropic='Anthropic API', openai='OpenAI', codex='Codex', gemini='Gemini', copilot='GitHub Copilot',
                 azure='Azure', remote='Other hosted models', litert='On this Mac (LiteRT)', mlx='On this Mac (MLX)',
                 llama='On this Mac (llama.cpp)', ollama='On this Mac (Ollama)')
LOCAL = ('litert', 'mlx', 'llama', 'ollama')
_DROP = re.compile(r'^(?:mlx|gguf|it|litert|lm|optiq|instruct|\d+bit)$', re.I)

def _words(m:str) -> str:
    "`claude-opus-5-5` gives `Claude Opus 5.5`, `gpt-6-astra` gives `GPT-6 Astra`, `Qwen3-4B` gives `Qwen 3 4B`."
    m = re.sub(r'-20\d{6}$', '', m)
    out = []
    for t in (t for t in re.split(r'[-_:\s]+', m) if t and not _DROP.match(t)):
        if t.isdigit() and out and re.fullmatch(r'[\d.]+', out[-1]) and '.' not in out[-1]: out[-1] += '.' + t
        elif re.fullmatch(r'[a-z]?[\d.]+[bmk]', t, re.I): out.append(t.upper())
        elif t.lower() == 'gpt': out.append('GPT')
        elif out and out[-1].startswith('GPT') and re.fullmatch(r'[\d.]+', t) and out[-1] == 'GPT': out[-1] += f'-{t}'
        else: out.append(re.sub(r'^([A-Za-z]+)(\d)', r'\1 \2', t[0].upper() + t[1:]))
    return ' '.join(out)

def model_label(value:str, provider:str='', repo:str='') -> str:
    "A name that people know for the model id `value`; `repo` is the download of a local model, if known."
    pre, _, m = value.rpartition('/')
    provider = provider or pre
    if provider in ('remote', 'azure'): return value   # a name that you gave in ramabana's config.toml
    name = _words(repo.rpartition('/')[2] if repo else m) or value
    if provider in ('claude', 'anthropic') and not name.startswith('Claude'): name = f'Claude {name}'
    return f'a local model: {name}' if provider in LOCAL else name

Each kind of id gets a plain name. A Claude Code tier gets the word Claude. A local model says so.

In [ ]:
test_eq(model_label('claude/sonnet'), 'Claude Sonnet')
test_eq(model_label('claude/claude-opus-5-5'), 'Claude Opus 5.5')
test_eq(model_label('anthropic/claude-haiku-4-5-20251001'), 'Claude Haiku 4.5')
test_eq(model_label('codex/gpt-6-astra'), 'GPT-6 Astra')
test_eq(model_label('openai/gpt-5.4-mini'), 'GPT-5.4 Mini')
test_eq(model_label('gemini/gemini-3.5-flash'), 'Gemini 3.5 Flash')
test_eq(model_label('qwen-4b', 'mlx', 'mlx-community/Qwen3.5-4B-MLX-4bit'), 'a local model: Qwen 3.5 4B')
test_eq(model_label('gemma-e4b', 'litert', 'litert-community/gemma-4-E4B-it-litert-lm'), 'a local model: Gemma 4 E4B')
test_eq(model_label('ollama/qwen3:8b'), 'a local model: Qwen 3 8B')
test_eq(model_label('my-model', 'azure'), 'my-model')

`catalog` lists the models that can run here, for the Settings page. With the `models` extra, the list is the full list of models. Each row gets a `name` from `model_label`. The list stays in memory for five minutes, because the look-up asks each provider. Without the extra, the list has only the models in `FALLBACK`.

In [ ]:
#| export
FALLBACK = ['claude/sonnet', 'claude/haiku', 'claude/opus']
_cat = {}

def _repos():
    "The download of each local model in the full model list, for the names."
    try:
        from ramabana.core import LOCAL as lr, MLX, LLAMA
        return {**lr, **MLX, **LLAMA}
    except Exception: return {}

def catalog(ttl:float=300.) -> list:
    "The models that can run here, as dicts with `value`, `name`, `label`, `provider` and `source`."
    if _cat.get('at', 0) > time.time() - ttl: return _cat['rows']
    try:
        from ramabana.core import available_models
        rows = available_models()
    except Exception: rows = [dict(value=m, label=m, provider='claude', source='rishi') for m in FALLBACK]
    repos = _repos()
    rows = [dict(r, name=model_label(r['value'], r.get('provider', ''), repos.get(r['value'], ''))) for r in rows]
    _cat.update(rows=rows, at=time.time())
    return rows

Without the `models` extra, the list has the three Claude tiers, each with a name.

In [ ]:
_cat.clear()
try: import ramabana; has_rama = True
except ImportError: has_rama = False
if not has_rama: test_eq([r['name'] for r in catalog()], ['Claude Sonnet', 'Claude Haiku', 'Claude Opus'])
assert all(r['name'] for r in catalog())

`Router` is the `ask` the app gives the tutor. Called as it is, it uses the default model. `for_(task)` gives the `ask` for one
task: the model the settings name for it, else the default.

In [ ]:
#| export
class Router:
    "The tutor's model: a default `ask`, and one model for each task that the settings name."
    def __init__(self, dflt, models=None):
        self.dflt, self.model = dflt, getattr(dflt, 'model', '')
        self.models = models or (lambda: __import__('abhyasa.voice', fromlist=['settings']).settings().get('models', {}))
    def __call__(self, prompt, sp=''): return self.dflt(prompt, sp)
    def name(self, task:str) -> str:
        "The model for `task`, or '' for the default."
        return (self.models() or {}).get(task) or ''
    def for_(self, task:str):
        "The `ask` for `task`."
        n = self.name(task)
        return model_ask(n) if n else self.dflt

def for_task(ask, task:str):
    "The `ask` for `task` when `ask` routes by task, else `ask` itself."
    return ask.for_(task) if hasattr(ask, 'for_') else ask

A router with no settings uses the default for every task. A task that names a model gets that model.

In [ ]:
dflt = Fake(dflt='default')
r = Router(dflt, models=lambda: {})
test_eq(r('x'), 'default')
test_is(r.for_('grade'), dflt)
named = Fake(dflt='named'); named.model = 'fast'
_asks['fast'] = named
r = Router(dflt, models=lambda: dict(grade='fast'))
test_eq(r.for_('grade')('x'), 'named')
test_eq(r.for_('question')('x'), 'default')
test_is(for_task(dflt, 'grade'), dflt)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()